# neural-playback — Phase 0 Validation
Tests: (1) TRIBE v2 loads on T4, (2) VRAM stays under 14GB during inference, (3) 30-second audio completes in under 5 minutes

In [ ]:
!pip install -q tribev2 nilearn ffmpeg-python soundfile

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import subprocess, numpy as np, tempfile, os, soundfile as sf

sr = 16000
duration = 30
t = np.linspace(0, duration, sr * duration, dtype=np.float32)
tone = (0.5 * np.sin(2 * np.pi * 440 * t)).astype(np.float32)
wav_path = "/tmp/test_tone.wav"
sf.write(wav_path, tone, sr)

mp4_path = "/tmp/test_tone.mp4"
result = subprocess.run(
    ["ffmpeg", "-y", "-i", wav_path,
     "-f", "lavfi", "-i", "color=c=black:s=320x240:r=1",
     "-shortest", "-c:v", "libx264", "-c:a", "aac", mp4_path],
    capture_output=True
)
print(f"Test MP4 created: {os.path.getsize(mp4_path)} bytes")

In [ ]:
from tribev2 import TribeModel
import torch, time

torch.cuda.reset_peak_memory_stats()
print("Loading model...")
t0 = time.time()
model = TribeModel.from_pretrained("facebook/tribev2", cache_folder="./cache")
load_time = time.time() - t0
vram_after_load = torch.cuda.max_memory_allocated() / 1e9
print(f"Model loaded in {load_time:.1f}s")
print(f"VRAM after load: {vram_after_load:.2f} GB")

In [ ]:
torch.cuda.reset_peak_memory_stats()
print("Running inference on 30s test tone...")
t0 = time.time()
df = model.get_events_dataframe(video_path="/tmp/test_tone.mp4")
preds, segments = model.predict(events=df)
inference_time = time.time() - t0
peak_vram = torch.cuda.max_memory_allocated() / 1e9
print(f"Inference time: {inference_time:.1f}s")
print(f"Peak VRAM: {peak_vram:.2f} GB")
print(f"Output shape: {preds.shape}")
print(f"Segments: {len(segments)}")

In [ ]:
import json

VRAM_LIMIT = 14.0
TIME_LIMIT = 300

results = {
    "vram_ok": bool(peak_vram < VRAM_LIMIT),
    "time_ok": bool(inference_time < TIME_LIMIT),
    "peak_vram_gb": round(float(peak_vram), 2),
    "inference_time_s": round(float(inference_time), 1),
    "output_shape": list(preds.shape),
}
print(json.dumps(results, indent=2))

if results["vram_ok"] and results["time_ok"]:
    print("\nVALIDATION PASSED - proceed to Phase 1")
else:
    if not results["vram_ok"]:
        print(f"\nVRAM EXCEEDED: {peak_vram:.2f} GB > {VRAM_LIMIT} GB - need Colab Pro (A100) or fp16 quantization")
    if not results["time_ok"]:
        print(f"\nTOO SLOW: {inference_time:.1f}s for 30s audio - full track will be impractical, need section-only processing")

## Next Steps
- If PASSED: proceed to Phase 1 core pipeline
- If FAILED on VRAM: document minimum GPU requirement, add `torch.cuda.amp.autocast()` half-precision path and retest
- If FAILED on time: implement section-only processing (30-60s segments) instead of full-track inference